Establish connection to Oracle database using SQLAlchemy with specified credentials

In [146]:
import pandas as pd #Imports pandas library for data manipulation and analysis.
from sqlalchemy import create_engine    #Imports SQLAlchemy function to create a connection engine for interacting with databases.

#Define Oracle database connection parameters including username, password, host, port, and service name.
user = "saptest"
password = "Corpdb#5aptest"
host = "corpdb.bhel.in"
port = "1521"
service = "ORCL"

#Creates a SQLAlchemy engine to establish a connection with the Oracle database using provided credentials.
engine = create_engine(
    f"oracle+oracledb://{user}:{password}@{host}:{port}/?service_name={service}"
)

Execute SQL query to retrieve data from the Oracle table and load it into a pandas DataFrame for further analysis.

In [147]:
query = """
SELECT *
FROM SAPTEST.PERF_FACT_MONTHLY_SNAPSHOT
"""

df = pd.read_sql(query, engine)

Display the first few rows of the DataFrame to quickly inspect the structure and contents of the loaded data.

In [148]:
df.head()

,snapshot_month,emp_id,gender,tenure_yy,age_yy,cadre,grade,unit,location,doj,...,prod_3m_slope,prod_volatility,prod_risk_flag,overtime_3m_avg,absence_3m_avg,cons_overtime_months,leave_spike_flag,burnout_risk_flag,pay_growth_trend,comp_stag_flag
0,2022-07-01,1257889,1,35.81,57,1,7,1010,2,1986-10-07,...,-0.09,3.34,0,12.87,1.00,0,1,1,1.38,0
1,2022-08-01,1257889,1,35.90,57,1,7,1010,2,1986-10-07,...,-1.32,3.96,0,9.67,1.00,0,0,0,1.38,0
2,2022-09-01,1257889,1,35.98,58,1,7,1010,2,1986-10-07,...,0.24,3.69,0,9.03,1.33,0,0,0,1.38,0
3,2022-10-01,1257889,1,36.06,58,1,7,1010,2,1986-10-07,...,0.36,3.99,0,7.13,1.00,0,0,0,0.00,0
4,2022-11-01,1257889,1,36.15,58,1,7,1010,2,1986-10-07,...,0.65,3.39,0,9.07,1.00,0,0,0,0.00,0


List all column names in the DataFrame to understand the available fields in the dataset.

In [149]:
df.columns

Index(['snapshot_month', 'emp_id', 'gender', 'tenure_yy', 'age_yy', 'cadre',
       'grade', 'unit', 'location', 'doj', 'dor', 'yy_since_prom',
       'stagflation', 'career_velocity', 'tasks_assigned', 'tasks_completed',
       'prod_rate', 'timeliness', 'error_rate', 'work_hours', 'utilise_rate',
       'produc_hours', 'high_prod_flag', 'rating', 'kpi', 'goal_perc',
       'mgr_rating', 'potential', 'perf_change', 'basic', 'bonus', 'incentive',
       'total', 'compa_ratio', 'sal_growth', 'engage_score', 'burn_score',
       'well_score', 'manager_score', 'culture_score', 'workload_score',
       'working_days', 'absence_days', 'sick_leaves', 'casual_leaves',
       'late_logins', 'early_exits', 'overtime_hh', 'train_hh_6m',
       'avg_train_score_6m', 'cert_count', 'prod_3m_avg', 'prod_3m_slope',
       'prod_volatility', 'prod_risk_flag', 'overtime_3m_avg',
       'absence_3m_avg', 'cons_overtime_months', 'leave_spike_flag',
       'burnout_risk_flag', 'pay_growth_trend', 'comp_st

Calculate the total number of missing (null) values in each column to assess data completeness.

In [150]:
na_counts = df.isna().sum()

# Show only columns where NA count > 0
na_counts = na_counts[na_counts > 0]
na_counts

yy_since_prom          7137
career_velocity           4
rating                10196
kpi                   10196
goal_perc             10196
mgr_rating            10196
potential             10196
perf_change           10196
engage_score          24089
burn_score            24089
well_score            24089
manager_score         24089
culture_score         24089
workload_score        24089
train_hh_6m           92655
avg_train_score_6m    92655
prod_3m_slope         39249
dtype: int64

Perform data cleaning by handling missing values using defaults, medians, and group-wise imputations to ensure completeness and consistency of features for analysis.

In [151]:
df2 = df.copy()
df2['yy_since_prom'] = df2['yy_since_prom'].fillna(df['tenure_yy'])
df2['rating'] = df2.groupby('grade')['rating'].transform(lambda x: x.fillna(x.median()))
df2['kpi'] = df2.groupby('grade')['kpi'].transform(lambda x: x.fillna(x.median()))
df2['goal_perc'] = df2.groupby('grade')['goal_perc'].transform(lambda x: x.fillna(x.median()))
df2['mgr_rating'] = df2.groupby('grade')['mgr_rating'].transform(lambda x: x.fillna(x.median()))
df2['potential'] = df2.groupby('grade')['potential'].transform(lambda x: x.fillna(x.median()))
df2['perf_change'] = df2['yy_since_prom'].fillna(0)
df2['train_hh_6m'] = df2['train_hh_6m'].fillna(0)
df2['avg_train_score_6m'] = df2['avg_train_score_6m'].fillna(0)
df2['prod_3m_slope'] = df2['prod_3m_slope'].fillna(0)
df2['career_velocity'] = df2['career_velocity'].fillna(0)
df2['engage_score'] = df2.groupby('unit')['engage_score'].transform(lambda x: x.fillna(x.median()))
df2['burn_score'] = df2.groupby('unit')['burn_score'].transform(lambda x: x.fillna(x.median()))
df2['well_score'] = df2.groupby('unit')['well_score'].transform(lambda x: x.fillna(x.median()))
df2['manager_score'] = df2.groupby('unit')['manager_score'].transform(lambda x: x.fillna(x.median()))
df2['culture_score'] = df2.groupby('unit')['culture_score'].transform(lambda x: x.fillna(x.median()))
df2['workload_score'] = df2.groupby('unit')['workload_score'].transform(lambda x: x.fillna(x.median()))

In [152]:
na_counts = df2.isna().sum()
# Show only columns where NA count > 0
na_counts = na_counts[na_counts > 0]
na_counts

Series([], dtype: int64)

Converting Colum Headers to Upper Case

In [153]:
df2.columns = df2.columns.str.upper()

Fetch Latest Dataframe (Snapshot Date = 2026-05-01) & Training Dataframe (Snapshot Date between 01-05-2021 to 01-12-2025)

In [154]:
latest_snapshot = pd.Timestamp("2026-05-01")

df_latest = df2[df2["SNAPSHOT_MONTH"] == latest_snapshot].copy()

df_train = df2[(df2["SNAPSHOT_MONTH"] <= "2026-04-01") & (df2["SNAPSHOT_MONTH"] >= "2021-05-01")].copy()

Define target variable for model training.

In [155]:
y = df_train["PROD_RISK_FLAG"]

In [156]:
df_train['GRADE'] = df_train['GRADE'].str.replace("T","9")

Identify highly correlated features (>0.8) to remove multicollinearity in the dataset.

In [157]:
import pandas as pd
import numpy as np

# correlation matrix
corr_matrix = df_train.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# find columns with correlation > threshold
threshold = 0.9

to_drop = [
    column for column in upper.columns
    if any(upper[column] > threshold)
]

print("Columns to drop:", to_drop)

Columns to drop: ['DOJ', 'TASKS_COMPLETED', 'PRODUC_HOURS', 'PERF_CHANGE', 'TOTAL', 'BURNOUT_RISK_FLAG']


In [158]:
df3 = df2.drop(columns=['AGE_YY', 'GRADE', 'DOJ', 'DOR', 'TASKS_COMPLETED', 'TIMELINESS', 'PRODUC_HOURS', 'PERF_CHANGE', 'TOTAL', 'CASUAL_LEAVES', 'PROD_3M_AVG', 'OVERTIME_3M_AVG', 'BURNOUT_RISK_FLAG'])

Remove non-feature columns to create input datasets for model training and prediction.

In [159]:
drop_cols = [
    "EMP_ID", "SNAPSHOT_MONTH",
    "PROD_RISK_FLAG", "HIGH_PROD_FLAG",'BURNOUT_RISK_FLAG',
    "PROD_RATE", "ERROR_RATE", "PROD_VOLATILITY",'PROD_3M_AVG','PROD_3M_SLOPE',
    "BASIC", "BONUS", "INCENTIVE",'TOTAL',
    "GENDER", "UNIT", "LOCATION", "CADRE", 'AGE_YY', 'GRADE', 'DOJ', 'DOR', 
    'TASKS_COMPLETED', 'TIMELINESS', 'PRODUC_HOURS', 'PERF_CHANGE', 'CASUAL_LEAVES', 'OVERTIME_3M_AVG'
]

X = df_train.drop(columns=drop_cols, errors="ignore")
X_latest = df_latest.drop(columns=drop_cols, errors="ignore")

Import libraries for data handling, database access, model training, and evaluation.

In [160]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score

Split data into training and test sets while preserving class distribution (Train = 75% / Test = 25%). 

In [161]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42, #Controls randomness
    stratify=y #Ensures class balance is preserved
)

Create a pipeline with scaling and logistic regression model for balanced classification.

In [162]:
pipe = Pipeline([
    ("scaler", StandardScaler()), #standardizes features
    ("model", LogisticRegression(  #create Logistic Regression Model
        class_weight="balanced",
        max_iter=2000,
        C=0.5
    ))
])

Train the model pipeline using the training data.

In [163]:
pipe.fit(X_train, y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.5


Generate predicted probabilities for test data.

In [164]:
pred_test = pipe.predict_proba(X_test)[:,1]

Calculate ROC AUC score to evaluate model performance.

In [165]:
auc = roc_auc_score(y_test, pred_test)

print("ROC AUC Score:", auc)

ROC AUC Score: 0.5460674520839972


Generate predicted attrition probabilities for the latest dataset.

In [166]:
prob_latest = pipe.predict_proba(X_latest)[:,1]
df_latest["PROD_RISK_PROB"] = prob_latest.round(6)

Define database credentials and create Oracle connection engine for data access.

In [167]:
user = "saptest"
password = "Corpdb#5aptest"
host = "corpdb.bhel.in"
port = "1521"
service = "ORCL"

engine = create_engine(
    f"oracle+oracledb://{user}:{password}@{host}:{port}/?service_name={service}"
)

In [168]:
df_latest.head()

,SNAPSHOT_MONTH,EMP_ID,GENDER,TENURE_YY,AGE_YY,CADRE,GRADE,UNIT,LOCATION,DOJ,...,PROD_VOLATILITY,PROD_RISK_FLAG,OVERTIME_3M_AVG,ABSENCE_3M_AVG,CONS_OVERTIME_MONTHS,LEAVE_SPIKE_FLAG,BURNOUT_RISK_FLAG,PAY_GROWTH_TREND,COMP_STAG_FLAG,PROD_RISK_PROB
703816,2026-05-01,6235638,1,13.27,34,1,3,1010,2,2013-02-23,...,2.88,0,1.47,1.33,0,0,0,0.94,0,0.427359
703818,2026-05-01,6078141,1,16.84,39,1,4,1014,27,2009-07-27,...,5.03,0,12.20,0.00,0,0,0,1.04,0,0.512214
703820,2026-05-01,6089844,1,16.71,43,1,5,1021,15,2009-09-15,...,4.16,0,4.60,3.00,0,0,0,1.39,0,0.438979
703822,2026-05-01,1643622,0,32.66,57,1,0,1011,9,1993-10-04,...,2.59,0,18.17,3.67,0,0,0,1.39,0,0.436213
703824,2026-05-01,6135900,1,15.29,39,1,4,1014,27,2011-02-16,...,4.77,0,4.67,2.67,0,0,0,1.73,0,0.473536


In [169]:
df_latest["PROD_RISK_PROB"].value_counts

<bound method IndexOpsMixin.value_counts of 703816    0.427359
703818    0.512214
703820    0.438979
703822    0.436213
703824    0.473536
            ...   
727895    0.500776
727897    0.518048
727899    0.491121
727901    0.351420
727903    0.464068
Name: PROD_RISK_PROB, Length: 11092, dtype: float64>

In [170]:
df_latest.columns

Index(['SNAPSHOT_MONTH', 'EMP_ID', 'GENDER', 'TENURE_YY', 'AGE_YY', 'CADRE',
       'GRADE', 'UNIT', 'LOCATION', 'DOJ', 'DOR', 'YY_SINCE_PROM',
       'STAGFLATION', 'CAREER_VELOCITY', 'TASKS_ASSIGNED', 'TASKS_COMPLETED',
       'PROD_RATE', 'TIMELINESS', 'ERROR_RATE', 'WORK_HOURS', 'UTILISE_RATE',
       'PRODUC_HOURS', 'HIGH_PROD_FLAG', 'RATING', 'KPI', 'GOAL_PERC',
       'MGR_RATING', 'POTENTIAL', 'PERF_CHANGE', 'BASIC', 'BONUS', 'INCENTIVE',
       'TOTAL', 'COMPA_RATIO', 'SAL_GROWTH', 'ENGAGE_SCORE', 'BURN_SCORE',
       'WELL_SCORE', 'MANAGER_SCORE', 'CULTURE_SCORE', 'WORKLOAD_SCORE',
       'WORKING_DAYS', 'ABSENCE_DAYS', 'SICK_LEAVES', 'CASUAL_LEAVES',
       'LATE_LOGINS', 'EARLY_EXITS', 'OVERTIME_HH', 'TRAIN_HH_6M',
       'AVG_TRAIN_SCORE_6M', 'CERT_COUNT', 'PROD_3M_AVG', 'PROD_3M_SLOPE',
       'PROD_VOLATILITY', 'PROD_RISK_FLAG', 'OVERTIME_3M_AVG',
       'ABSENCE_3M_AVG', 'CONS_OVERTIME_MONTHS', 'LEAVE_SPIKE_FLAG',
       'BURNOUT_RISK_FLAG', 'PAY_GROWTH_TREND', 'COMP_ST

In [171]:
from sklearn.preprocessing import MinMaxScaler

cols = [
    'TRAIN_HH_6M',
    'ENGAGE_SCORE',
    'CAREER_VELOCITY',
    'PROD_RATE',
    'PROD_RISK_PROB'
]

scaler = MinMaxScaler()

scaled = scaler.fit_transform(df_latest[cols])

In [172]:
scaled_df = pd.DataFrame(
    scaled,
    columns=cols
)

In [173]:
scaled_df.describe

<bound method NDFrame.describe of        TRAIN_HH_6M  ENGAGE_SCORE  CAREER_VELOCITY  PROD_RATE  PROD_RISK_PROB
0         0.250350      0.707880         0.291139   0.373057        0.323449
1         0.085941      0.682745         0.303797   0.522021        0.579679
2         0.178421      0.680027         0.379747   0.627202        0.358537
3         0.036899      0.681386         0.265823   0.846373        0.350185
4         0.160206      0.682745         0.329114   0.953627        0.462886
...            ...           ...              ...        ...             ...
11087     0.205511      0.683424         0.303797   0.749741        0.545140
11088     0.135451      0.707880         0.253165   0.730570        0.597295
11089     0.196170      0.707880         0.329114   0.481088        0.515986
11090     0.099486      0.681386         0.354430   0.295596        0.094143
11091     0.000000      0.707880         0.291139   0.945596        0.434296

[11092 rows x 5 columns]>

In [174]:
scaled_df.isna().sum()

TRAIN_HH_6M        0
ENGAGE_SCORE       0
CAREER_VELOCITY    0
PROD_RATE          0
PROD_RISK_PROB     0
dtype: int64

In [175]:
df_latest.count()

SNAPSHOT_MONTH       11092
EMP_ID               11092
GENDER               11092
TENURE_YY            11092
AGE_YY               11092
                     ...  
LEAVE_SPIKE_FLAG     11092
BURNOUT_RISK_FLAG    11092
PAY_GROWTH_TREND     11092
COMP_STAG_FLAG       11092
PROD_RISK_PROB       11092
Length: 63, dtype: int64

In [176]:
scaled_df['READINESS_INDEX'] = (

    0.25 * scaled_df['ENGAGE_SCORE']

     + 0.25 * scaled_df['PROD_RATE']

    + 0.20 * scaled_df['TRAIN_HH_6M']

    + 0.15 * scaled_df['CAREER_VELOCITY']

    + 0.15 * (1 - scaled_df['PROD_RISK_PROB'])

) * 100

In [177]:
scaled_df.describe

<bound method NDFrame.describe of        TRAIN_HH_6M  ENGAGE_SCORE  CAREER_VELOCITY  PROD_RATE  PROD_RISK_PROB  \
0         0.250350      0.707880         0.291139   0.373057        0.323449   
1         0.085941      0.682745         0.303797   0.522021        0.579679   
2         0.178421      0.680027         0.379747   0.627202        0.358537   
3         0.036899      0.681386         0.265823   0.846373        0.350185   
4         0.160206      0.682745         0.329114   0.953627        0.462886   
...            ...           ...              ...        ...             ...   
11087     0.205511      0.683424         0.303797   0.749741        0.545140   
11088     0.135451      0.707880         0.253165   0.730570        0.597295   
11089     0.196170      0.707880         0.329114   0.481088        0.515986   
11090     0.099486      0.681386         0.354430   0.295596        0.094143   
11091     0.000000      0.707880         0.291139   0.945596        0.434296   

     

In [178]:
df_latest['READINESS_INDEX'] = scaled_df['READINESS_INDEX'].values

Write predicted results to Oracle table in batches for efficient database storage.

In [179]:
df_latest.to_sql(
    name="PERF_PROD_RISK_PROBAB",
    schema="SAPTEST",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000
)

C:\Users\asad\AppData\Local\Temp\ipykernel_20192\2315138634.py:1: UserWarning: The provided table name 'PERF_PROD_RISK_PROBAB' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_latest.to_sql(


-3

Extract model coefficients and map them to features for interpretability.

In [180]:
import pandas as pd
import numpy as np

# Extract trained logistic regression model
model = pipe.named_steps["model"]

# Create dataframe of coefficients
coef_df = pd.DataFrame({
    "FEATURE": X.columns,
    "COEFF": model.coef_[0].round(8)
})

print(coef_df.head())

           FEATURE     COEFF
0        TENURE_YY -0.068942
1    YY_SINCE_PROM  0.022990
2      STAGFLATION -0.030017
3  CAREER_VELOCITY -0.002431
4   TASKS_ASSIGNED -0.001528


Save feature coefficients to Oracle table for attrition driver analysis.

In [181]:
coef_df.to_sql(
    name="PERF_PROD_RISK_COEFF",
    schema="SAPTEST",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000
)

C:\Users\asad\AppData\Local\Temp\ipykernel_20192\1210651534.py:1: UserWarning: The provided table name 'PERF_PROD_RISK_COEFF' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  coef_df.to_sql(


-1